# CineInfini — Deployment tests

Validates the deployment script (`deploy_cineinfini.py`) without
making any external network calls. Uses the script's `--dry-run` mode.

**Safe to run anywhere** — no GitHub, no PyPI, no Zenodo calls.


## 1. Locate the deploy script

In [ ]:
import os, sys, subprocess
from pathlib import Path

repo = Path(os.environ.get('CINEINFINI_REPO', '.'))
deploy_script = repo / 'deploy_cineinfini.py'
print(f"Deploy script: {deploy_script}")
print(f"Exists: {deploy_script.exists()}")
print(f"Size:   {deploy_script.stat().st_size if deploy_script.exists() else 0} bytes")


## 2. Pre-flight: does the version bump compile?

In [ ]:
init_file = repo / 'src' / 'cineinfini' / '__init__.py'
print(init_file.read_text().split('__version__')[1].split('\n')[0])


## 3. Run the test suite (gate for deployment)

In [ ]:
env = {**os.environ, 'PYTHONPATH': f'{repo}/src'}
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/', '-q',
     '-m', 'not integration and not slow'],
    cwd=repo, env=env, capture_output=True, text=True, timeout=300,
)
last_lines = result.stdout.splitlines()[-5:]
print('\n'.join(last_lines))
assert result.returncode == 0, "Tests failed — deploy would be blocked"
print("✓ Tests green; deploy gate would pass")


## 4. Verify required release artifacts exist

In [ ]:
required = [
    'README.md', 'CHANGELOG.md', 'CITATION.cff', 'LICENSE',
    'pyproject.toml', 'src/cineinfini/__init__.py',
    'cfg/config.yaml', 'cfg/config.test.yaml',
    'cfg/profiles/realtime.yaml', 'cfg/profiles/academic.yaml',
    'cfg/profiles/postproduction.yaml', 'cfg/profiles/ultralight.yaml',
    'cfg/profiles/low_memory.yaml',
    'docs/USER_MANUAL.md', 'docs/INSTALLATION.md', 'docs/QUICKSTART.md',
    'docs/STATUS.md', 'docs/MISSING_ASSETS.md',
    'docs/benchmarking/COMPARISON.md',
    'docs/benchmarking/COMPETITORS.md',
    'docs/benchmarking/PROFILES.md',
]
missing = [f for f in required if not (repo / f).exists()]
for f in required:
    mark = '✓' if (repo / f).exists() else '✗'
    size = (repo / f).stat().st_size if (repo / f).exists() else 0
    print(f"  {mark} {f:55s} {size:8d} B")
assert not missing, f"Missing release artifacts: {missing}"
print(f"\n✓ All {len(required)} required files present")


## 5. Verify version consistency across files

In [ ]:
import re
init_text = (repo / 'src/cineinfini/__init__.py').read_text()
init_v = re.search(r'__version__\s*=\s*"([^"]+)"', init_text).group(1)

cit_text = (repo / 'CITATION.cff').read_text()
cit_v = re.search(r'^version:\s*([^\s]+)', cit_text, re.MULTILINE).group(1)

readme_text = (repo / 'README.md').read_text()
readme_match = re.search(r'version\s*=\s*\{?([\d.]+)\}?', readme_text)
readme_v = readme_match.group(1) if readme_match else "n/a"

print(f"  __init__.py:    {init_v}")
print(f"  CITATION.cff:   {cit_v}")
print(f"  README.md:      {readme_v}")
assert init_v == cit_v, f"Version drift: {init_v} != {cit_v}"
print(f"\n✓ Versions consistent: {init_v}")


## 6. Build a wheel locally (no upload)

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'build', '--quiet'],
    capture_output=True, text=True,
)
result = subprocess.run(
    [sys.executable, '-m', 'build', '--sdist', '--wheel', '--outdir', '/tmp/dist'],
    cwd=repo, capture_output=True, text=True, timeout=120,
)
print('--- stdout (last 1500 chars) ---')
print(result.stdout[-1500:])
if result.returncode:
    print('--- stderr ---')
    print(result.stderr[-1500:])
print(f"\nExit code: {result.returncode}")


## What this notebook validates

If every cell is green:
1. ✅ Deploy script is in place
2. ✅ Version is consistent across __init__.py / CITATION.cff / README.md
3. ✅ All 232 tests pass (the pre-deploy gate)
4. ✅ Every required release artifact (README, CHANGELOG, profiles, docs) exists
5. ✅ The package builds cleanly with `python -m build`

After this you can safely run `python deploy_cineinfini.py --dry-run` to
preview the GitHub release + PyPI publish without actually pushing.

To do the real deploy: re-run without `--dry-run` and supply
`GITHUB_KEY` + `PYPI_API_TOKEN` in the environment.
